In [211]:
import time
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import csv
import pandas as pd
import numpy as np

In [94]:
options = Options()
driver = webdriver.Chrome(options=options)

links = []
base = 'https://ekb.cian.ru/cat.php?deal_type=sale&engine_version=2&object_type%5B0%5D=1&offer_type=flat&region=4743'
rooms_urls = ['room1', 'room2', 'room3', 'room4', 'room5', 'room6', 'room7', 'room9']

def scrape_links():
    global links
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')
    for a in soup.find_all('a', href=True):
        href = a['href']
        if '/sale/flat/' in href and href not in links:
            links.append(href)

def scrape_num_flats():
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')
    num_to_sp = soup.find_all('h5')[0].get_text()
    if num_to_sp.split()[2].isdigit():
        num = int(num_to_sp.split()[1] + num_to_sp.split()[2])
    else:
        num = int(num_to_sp.split()[1])
    return num

for rooms in rooms_urls: 
    url_room = f'{base}&{rooms}=1'
    driver.get(url_room)
    n = scrape_num_flats()
    print(f'{rooms} - {n} обьявлений')
    pages = n//28 + 1
    if pages < 54: #после 54 страницы на циане заканчивается пагинация
        for p in range(1, pages+1):
            pag_url = f'{url_room}&p={p}'
            driver.get(pag_url)
            scrape_links()
    else:
        for p in range(1, 54+1):
            pag_url = f'{url_room}&p={p}'
            driver.get(pag_url)
            scrape_links()
        for _ in range(1, pages - 54 + 1):
            try:
                button = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, "a[class*='more-button']")))
                driver.execute_script('arguments[0].scrollIntoView(true);', button)
                button.click()
            except:
                break
        scrape_links()




room1 - 1714 обьявлений


Exception ignored in: <function Service.__del__ at 0x0000019CBC458220>
Traceback (most recent call last):
  File "C:\Users\anton\anaconda3\Lib\site-packages\selenium\webdriver\common\service.py", line 196, in __del__
    self.stop()
  File "C:\Users\anton\anaconda3\Lib\site-packages\selenium\webdriver\common\service.py", line 152, in stop
    self.send_remote_shutdown_command()
  File "C:\Users\anton\anaconda3\Lib\site-packages\selenium\webdriver\common\service.py", line 137, in send_remote_shutdown_command
    if not self.is_connectable():
  File "C:\Users\anton\anaconda3\Lib\site-packages\selenium\webdriver\common\service.py", line 126, in is_connectable
    return utils.is_connectable(self.port)
  File "C:\Users\anton\anaconda3\Lib\site-packages\selenium\webdriver\common\utils.py", line 117, in is_connectable
    socket_ = socket.create_connection((host, port), 1)
  File "C:\Users\anton\anaconda3\Lib\socket.py", line 856, in create_connection
    exceptions.clear()  # raise only the

TypeError: object of type 'NoneType' has no len()

In [57]:
with open('cian_links.csv', 'w', encoding='utf-8') as f:
    wr = csv.writer(f)
    for link in links:
        wr.writerow([link])
driver.quit()

# Парсинг каждой хаты

In [208]:
links = pd.read_csv('cian_links.csv', header=None).iloc[:, 0].tolist() 
len(links)

5967

In [212]:
options = Options()
driver = webdriver.Chrome(options=options)

def parse_flat(link):
    driver.get(link)
    driver.execute_script("window.scrollTo({top: document.body.scrollHeight, behavior: 'smooth'});")
    time.sleep(2)
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')
    flat_dict = {}

    flat_dict['Ссылка'] = link

    #Заголовок
    try:
        h1 = soup.find('h1').text.strip()
        flat_dict['Заголовок'] = h1
    except:
        flat_dict['Заголовок'] = np.nan

    #цена
    try:
        price = soup.find('div', {'data-name': 'PriceInfo'}).find('span').text.strip()
        flat_dict['Цена'] = price
    except:
        flat_dict['Цена'] = np.nan

    #Оценка циана
    try:
        zian_value = soup.find('div', {'data-name': 'PricesBlock'}).find('div', class_=lambda c:'item' in c).find('span').text.strip()
        flat_dict['Оценка циана'] = zian_value
    except:
        flat_dict['Оценка циана'] = np.nan

    #ЖК
    try:
        jk = soup.find('div', {'data-name': 'ParentNew'}).find('a').text.strip()
        flat_dict['ЖК'] = jk
    except:
        flat_dict['ЖК'] = np.nan

    #Агенство
    try:
        rieltor = soup.find('div', {'data-name': 'AgencyBrandingAsideCardComponent'}).find('a').find('span').text.strip()
        flat_dict['Агенство'] = rieltor
    except:
        flat_dict['Агенство'] = np.nan

    #всякие шняги
    try:
        shniagi = soup.find_all('div', {'data-name': 'ObjectFactoidsItem'})
        for sh in shniagi:
            t = sh.find('div', class_=lambda c:'text' in c)
            name, val = t.find_all('span')
            flat_dict[name.text.strip()] = val.text.strip()
    except:
        pass

    #метро
    try:
        all_st = []
        metros = soup.find('ul', {'data-name': 'UndergroundList'})
        for li in metros.find_all('li', {'data-name': 'UndergroundItem'}):
            station = li.find('a', class_=lambda c:'underground_link' in c).text.strip()
            tim = li.find('span', class_=lambda c:'underground_time' in c)
            time_text = tim.text.strip()
            paths = tim.find('svg').find_all('path')
            if len(paths) == 1:
                way = 'На машине'
            else:
                way = 'Пешком'
            
            all_st.append(f'{station} - {time_text} - {way}')
        flat_dict['Метро'] = '|'.join(all_st)
    except:
        flat_dict['Метро'] = np.nan

    #адрес
    try:
        address = []
        for part in soup.find('div', {'data-name': 'AddressContainer'}).find_all('a'):
            address.append(part.text.strip())
        flat_dict['Адрес'] = '|'.join(address)
    except:
        flat_dict['Адрес'] = np.nan

    #фото
    try:
        scroller = soup.find('div', {'data-name': 'Scroller'})
        photos = []
        for im in scroller.find_all('img', src=True):
            photos.append(im['src'])

        flat_dict['Фото'] = ' | '.join(photos)
    except:
        flat_dict['Фото'] = np.nan

    #описание
    try:
        descr = soup.find('div', {'data-name': 'Description'})
        desc = descr.find('span').text.strip()
        flat_dict['Описание'] = desc

    except:
        flat_dict['Описание'] = np.nan

    #пометки
    try:
        lab = soup.find('div', {'data-name': 'LabelsLayoutNew'}).find_all('span', class_=lambda c:'text' in c)
        for l in lab:
            if l.text.strip():
                flat_dict[l.text.strip()] = 1
    except:
        pass

    #условие сделки
    try:
        info = soup.find('div', {'data-name': 'OfferFactsInSidebar'}).find_all('div', {'data-name': 'OfferFactItem'})
        usl = info[1].find_all('span')[1].text.strip()
        flat_dict['Условие сделки'] = usl
    except:
        flat_dict['Условие сделки'] = np.nan

    #ипотека
    try:
        info = soup.find('div', {'data-name': 'OfferFactsInSidebar'}).find_all('div', {'data-name': 'OfferFactItem'})
        ipoteka = info[2].find_all('span')[1].text.strip()
        flat_dict['Ипотека'] = ipoteka
    except:
        flat_dict['Ипотека'] = np.nan

    #тонна инфы разной 
    try:
        items = soup.find('div', {'data-name': 'CardSectionNew'}).find_all('div', {'data-name': 'OfferSummaryInfoItem'})
        for i in items:
            name,value = i.find_all('p')
            flat_dict[name.text.strip()] = value.text.strip()
    except:
        pass


    #Цены квартир рядом
    try:
        others = soup.find('div', {'data-name': 'OfferHistoryContainer'}).find_all('article', {'data-name': 'NarrowOfferRow'})
        prices_others = []
        for o in others:
            pr = o.find('div',{'data-name': 'Price'}).find_all('span')[1].text.strip()
            prices_others.append(pr)
        flat_dict['Цены квартир рядом (за квадрат)'] = '|'.join(prices_others)
    except:
        flat_dict['Цены квартир рядом (за квадрат)'] = np.nan


    return flat_dict

al_flats = []
for_parse_list = links[:20]
for i, l in enumerate(for_parse_list, 1):
    print(f'Обработка {i}/{len(for_parse_list)}')
    try:
        data = parse_flat(l)
        al_flats.append(data)
    except:
        print(f'Ошибка при парсинге {l}')

df = pd.DataFrame(al_flats)
df

Обработка 1/20
Обработка 2/20
Обработка 3/20
Обработка 4/20
Обработка 5/20
Обработка 6/20
Обработка 7/20
Обработка 8/20
Обработка 9/20
Обработка 10/20
Обработка 11/20
Обработка 12/20
Обработка 13/20
Обработка 14/20
Обработка 15/20
Обработка 16/20
Обработка 17/20
Обработка 18/20
Обработка 19/20
Обработка 20/20


,Ссылка,Заголовок,Цена,Оценка циана,ЖК,Агенство,Общая площадь,Жилая площадь,Площадь кухни,Этаж,...,Цены квартир рядом (за квадрат),С мебелью,Балкон/лоджия,Продаётся с мебелью,Газоснабжение,Материнский капитал при покупке,О подъезде,Только на Циан,Проверено в Росреестре,Строительная серия
0,https://ekb.cian.ru/sale/flat/326262605/?conte...,"Продается 1-комн. квартира, 40,6 м²",4 466 000 ₽,"4,2 млн ₽",ЖК «Рудный»,Корпорация Маяк - отдел недвижимости,"40,6 м²","19,1 м²","9,5 м²",1 из 14,...,94 850 ₽/м²|110 972 ₽/м²|104 729 ₽/м²,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,https://ekb.cian.ru/sale/flat/329418752/?conte...,"Продается 1-комн. квартира, 40,8 м²",7 650 000 ₽,"6,4 млн ₽",ЖК «Eleven (Элевен)»,Ориентир. Недвижимость,"40,8 м²","18,7 м²","13,5 м²",18 из 32,...,160 965 ₽/м²|140 575 ₽/м²|169 318 ₽/м²,1.0,1 лоджия,Да,NaN,NaN,NaN,NaN,NaN,NaN
2,https://ekb.cian.ru/sale/flat/330568317/?conte...,"Продается 1-комн. квартира, 31 м²",2 800 000 ₽,"3,5 млн ₽",NaN,Дом Недвижимости,31 м²,14 м²,6 м²,1 из 5,...,97 674 ₽/м²|82 702 ₽/м²|94 966 ₽/м²,1.0,NaN,Да,Центральное,Не использовался,NaN,NaN,NaN,NaN
3,https://ekb.cian.ru/sale/flat/330865186/?conte...,"Продается 1-комн. квартира, 39,2 м²",5 200 000 ₽,"6,4 млн ₽",NaN,Виталия,"39,2 м²","17,2 м²","10,5 м²",13 из 13,...,113 414 ₽/м²|131 199 ₽/м²|121 245 ₽/м²,1.0,1 лоджия,Да,NaN,Не использовался,NaN,NaN,NaN,NaN
4,https://ekb.cian.ru/sale/flat/330267840/?conte...,"Продается 1-комн. квартира, 28,2 м²",4 100 000 ₽,"3,6 млн ₽",NaN,АЗБУКА НЕДВИЖИМОСТИ,"28,2 м²","15,4 м²",6 м²,9 из 9,...,105 354 ₽/м²|118 329 ₽/м²|109 166 ₽/м²,1.0,1 балкон,Да,Центральное,NaN,NaN,NaN,NaN,NaN
5,https://ekb.cian.ru/sale/flat/330245812/?conte...,"Продается 1-комн. квартира, 31,5 м²",3 800 000 ₽,"3,6 млн ₽",NaN,NaN,"31,5 м²","16,5 м²",7 м²,7 из 14,...,106 880 ₽/м²|132 022 ₽/м²|94 878 ₽/м²,1.0,1 лоджия,Да,NaN,Не использовался,Есть консьерж,NaN,NaN,NaN
6,https://ekb.cian.ru/sale/flat/330350157/?conte...,"Продается 1-комн. квартира, 34,4 м²",4 590 000 ₽,"4,2 млн ₽",ЖК «Садовый»,Агентство Солдатовой,"34,4 м²",15 м²,9 м²,8 из 14,...,110 507 ₽/м²|103 603 ₽/м²|111 130 ₽/м²,NaN,1 лоджия,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,https://ekb.cian.ru/sale/flat/329689564/?conte...,"Продается 1-комн. квартира, 34 м²",5 250 000 ₽,"4,2 млн ₽",NaN,АЗБУКА НЕДВИЖИМОСТИ,34 м²,18 м²,9 м²,6 из 12,...,167 630 ₽/м²|171 428 ₽/м²|153 846 ₽/м²,1.0,1 лоджия,Да,NaN,NaN,NaN,NaN,NaN,NaN
8,https://ekb.cian.ru/sale/flat/330807701/?conte...,"Продается 1-комн. квартира, 31,5 м²",3 800 000 ₽,"4,1 млн ₽",ЖК «Хрустальные Ключи»,Метражи,"31,5 м²","12,5 м²","12,5 м²",15 из 18,...,140 038 ₽/м²|100 993 ₽/м²|116 357 ₽/м²,NaN,1 балкон,NaN,Центральное,NaN,NaN,NaN,NaN,NaN
9,https://ekb.cian.ru/sale/flat/329898310/?conte...,"Продается 1-комн. квартира, 35 м²",4 190 000 ₽,"4,6 млн ₽",NaN,Новосел Краснолесье,35 м²,19 м²,9 м²,16 из 16,...,119 827 ₽/м²|100 000 ₽/м²|115 578 ₽/м²,NaN,1 балкон,NaN,Центральное,NaN,Есть мусоропровод,NaN,NaN,NaN


In [213]:
df.to_csv('hati20.csv', index = False)